# Colab runner – ML generalization for leakage-induced soil fluidization

Runs the complete analysis (≈ 25–40 min on a free Colab CPU with the default 30 Optuna trials; every fold is cached, so a disconnect only costs the unfinished fold).

**Before running:** put the folder `fluidization-ml-generalization` (this repository) inside `MyDrive/Fluidization/` on Google Drive. The raw data file `DataP.csv` ships in `data/raw/`; if you keep your own copy in `MyDrive/Fluidization/DataP.csv` it is used instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, pathlib
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/Fluidization')
REPO = DRIVE_DIR / 'fluidization-ml-generalization'
assert REPO.exists(), f'Put the repository folder in {DRIVE_DIR}'
os.chdir(REPO)
own = DRIVE_DIR / 'DataP.csv'
if own.exists():
    os.environ['FLUID_DATA_CSV'] = str(own); print('using', own)
os.environ['FLUID_AF_UNIT'] = 'cm2'   # confirmed: Af is in cm^2
print('repo:', REPO)

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .
!python -m pytest -q tests

## Full pipeline
`--n-trials 30` is the setting used for the manuscript. Results are written to `results/` (tables, figures, predictions) on Drive.

In [ ]:
!python scripts/run_all.py --n-trials 30

## Run the steps one by one (optional)

In [ ]:
# !python scripts/01_audit_data.py            # source-data audit: d50, units, non-monotonic records
# !python scripts/02_run_protocols.py --n-trials 30   # nested grouped CV for all protocols/models (cached per fold)
# !python scripts/03_summarize_tables.py
# !python scripts/04_recalibration.py         # forward-chaining / LOBO / in-sample
# !python scripts/05_bootstrap_shap_pdp.py    # bed-level bootstrap, SHAP, partial dependence
# !python scripts/08_powerlaw_exponents.py
# !python scripts/06_make_figures.py
# !python scripts/07_make_manuscript_tables.py
# !python scripts/09_compare_with_reference.py

## Look at the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/tables/T_protocol_summary.csv').round(3))
for f in sorted(pathlib.Path('results/figures').glob('fig*.png')):
    display(Image(str(f), width=700))